# SetWise MM-Fit + Whales Merged Dataset EDA

This notebook audits the merged dataset produced by `prepare.py`:

- `prepared/setwise_mmfit_whales_8to16_50hz_512.npz`
- `prepared/setwise_mmfit_whales_8to16_50hz_512_metadata.json`

The goal is to verify the data contract before model training: source counts, split policy, exercise coverage, rep-count supervision, duration/crop behavior, normalization, and simple count baselines.

In [ ]:
from __future__ import annotations

import json
import math
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

plt.style.use("default")
np.set_printoptions(precision=4, suppress=True)

In [ ]:
STEM = "setwise_mmfit_whales_8to16_50hz_512"
CANDIDATE_PREPARED_DIRS = [
    Path("prepared"),
    Path("../prepared"),
    Path.cwd() / "prepared",
    Path.cwd().parent / "prepared",
    Path("/content/drive/MyDrive/SetwiseKineticDatasets/prepared"),
]

prepared_dir = next(
    (candidate for candidate in CANDIDATE_PREPARED_DIRS if (candidate / f"{STEM}.npz").exists()),
    None,
)
assert prepared_dir is not None, "Missing prepared dataset. Run `python prepare.py --no-mount-drive` first."

npz_path = prepared_dir / f"{STEM}.npz"
metadata_path = prepared_dir / f"{STEM}_metadata.json"
assert metadata_path.exists(), f"Missing metadata: {metadata_path}"

raw = np.load(npz_path, allow_pickle=False)
with metadata_path.open("r") as f:
    metadata = json.load(f)

print(f"Loaded: {npz_path.resolve()}")
print(f"Metadata: {metadata_path.resolve()}")
print(f"Arrays: {len(raw.files)}")
print(raw.files)

In [ ]:
def as_str_array(name: str) -> np.ndarray:
    return np.asarray(raw[name]).astype(str)

X = raw["X"]
source = as_str_array("source")
split = as_str_array("split")
exercise_name = as_str_array("exercise_name")
original_exercise_name = as_str_array("original_exercise_name")
reps = raw["reps"].astype(int)
rep_supervised = raw["rep_supervised"].astype(bool)
classification_supervised = raw["classification_supervised"].astype(bool)
rep_class = raw["rep_class_8to16"].astype(int)
duration_s = raw["duration_s"].astype(float)
lengths_50hz = raw["lengths_50hz"].astype(int)
active_crop_used = raw["active_crop_used"].astype(bool)
feature_names = as_str_array("feature_names")
label_names = as_str_array("label_names")

print("X shape:", X.shape)
print("dtype:", X.dtype)
print("target_hz:", float(raw["target_hz"]))
print("model_length:", int(raw["model_length"]))
print("rep range:", int(raw["rep_min"]), "to", int(raw["rep_max"]))
print("feature_names:", feature_names.tolist())
print("classification supervised:", int(classification_supervised.sum()), "/", len(classification_supervised))
print("rep supervised:", int(rep_supervised.sum()), "/", len(rep_supervised))

In [ ]:
def counter_table(counter: Counter, headers=("Value", "Count")) -> str:
    rows = [f"| {headers[0]} | {headers[1]} |", "| --- | ---: |"]
    def key_fn(item):
        key = item[0]
        if isinstance(key, (int, np.integer)):
            return (0, int(key))
        text = str(key)
        return (0, int(text)) if text.lstrip("-").isdigit() else (1, text)
    for key, value in sorted(counter.items(), key=key_fn):
        rows.append(f"| `{key}` | {value} |")
    return "\n".join(rows)


def cross_table(row_values, col_values, row_name="row", col_name="col") -> str:
    row_values = [str(v) for v in row_values]
    col_values = [str(v) for v in col_values]
    rows = sorted(set(row_values), key=str)
    cols = sorted(set(col_values), key=str)
    counts = defaultdict(Counter)
    for r, c in zip(row_values, col_values):
        counts[r][c] += 1
    md = ["| " + row_name + " | " + " | ".join(cols) + " |", "| --- | " + " | ".join(["---:"] * len(cols)) + " |"]
    for r in rows:
        md.append("| `" + r + "` | " + " | ".join(str(counts[r][c]) for c in cols) + " |")
    return "\n".join(md)


def metric_summary(y_true, y_pred) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    err = np.abs(y_pred - y_true)
    return {
        "mae": float(err.mean()),
        "exact": float((err == 0).mean()),
        "within_1": float((err <= 1).mean()),
        "within_2": float((err <= 2).mean()),
    }


def metrics_table(rows: dict[str, dict[str, float]]) -> str:
    md = ["| Baseline | MAE | Exact | Within 1 | Within 2 |", "| --- | ---: | ---: | ---: | ---: |"]
    for name, m in rows.items():
        md.append(
            f"| {name} | {m['mae']:.3f} | {m['exact']:.2%} | {m['within_1']:.2%} | {m['within_2']:.2%} |"
        )
    return "\n".join(md)

## Dataset Contract

In [ ]:
display(Markdown(counter_table(Counter(source), ("Source", "Samples"))))
display(Markdown(counter_table(Counter(split), ("Split", "Samples"))))
display(Markdown(cross_table(source, split, "source", "split")))

expected_shape = tuple(metadata["tensor_shape"])
assert X.shape == expected_shape, (X.shape, expected_shape)
assert metadata["source_counts"] == {"mmfit": 616, "whales": 164}
assert int(rep_supervised.sum()) == metadata["rep_supervised_count"]
print("Metadata contract checks passed.")

## Exercise Coverage

In [ ]:
display(Markdown(counter_table(Counter(exercise_name), ("Canonical exercise", "Samples"))))
display(Markdown(cross_table(exercise_name, source, "exercise", "source")))

whales_mask = source == "whales"
mmfit_mask = source == "mmfit"
print("MM-Fit canonical exercises:", sorted(set(exercise_name[mmfit_mask])))
print("Whales canonical exercises:", sorted(set(exercise_name[whales_mask])))

## Rep Supervision Audit

In [ ]:
supervised_reps = reps[rep_supervised]
expected_counts = {8: 15, 9: 2, 10: 35, 11: 7, 12: 41, 13: 3, 14: 6, 15: 12, 16: 10}
actual_counts = Counter(supervised_reps.tolist())
assert actual_counts == expected_counts, actual_counts
assert set(rep_class[rep_supervised].tolist()) == set(range(9))
assert np.all(source[rep_supervised] == "whales")

display(Markdown(counter_table(actual_counts, ("Reps", "Whales supervised sets"))))
display(Markdown(cross_table(reps[rep_supervised], split[rep_supervised], "reps", "split")))
print("Unsupervised count labels retained for audit:", int((~rep_supervised).sum()))

## Count Baselines for 8-16 Whales

In [ ]:
y = reps[rep_supervised]
majority_rep = Counter(y.tolist()).most_common(1)[0][0]
rows = {
    "Always 10": metric_summary(y, np.full_like(y, 10)),
    f"Majority / always {majority_rep}": metric_summary(y, np.full_like(y, majority_rep)),
    "Rounded mean": metric_summary(y, np.full_like(y, int(round(float(y.mean()))))),
}
display(Markdown(metrics_table(rows)))

for split_name in ["train", "val", "test"]:
    mask = rep_supervised & (split == split_name)
    if mask.sum() == 0:
        continue
    y_split = reps[mask]
    split_rows = {
        "Always 10": metric_summary(y_split, np.full_like(y_split, 10)),
        f"Always {majority_rep}": metric_summary(y_split, np.full_like(y_split, majority_rep)),
    }
    display(Markdown(f"### {split_name} ({int(mask.sum())} samples)\n" + metrics_table(split_rows)))

## Durations, Lengths, and Whales Active Cropping

In [ ]:
def describe(values):
    values = np.asarray(values, dtype=float)
    return {
        "n": int(values.size),
        "min": float(np.min(values)),
        "p25": float(np.percentile(values, 25)),
        "median": float(np.median(values)),
        "p75": float(np.percentile(values, 75)),
        "max": float(np.max(values)),
        "mean": float(np.mean(values)),
    }

for src in ["mmfit", "whales"]:
    mask = source == src
    print(src, "duration_s", describe(duration_s[mask]))
    print(src, "lengths_50hz", describe(lengths_50hz[mask]))

whales_mask = source == "whales"
print("Whales active cropped:", int(active_crop_used[whales_mask].sum()), "/", int(whales_mask.sum()))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(duration_s[mmfit_mask], bins=30, alpha=0.7, label="MM-Fit")
axes[0].hist(duration_s[whales_mask], bins=30, alpha=0.7, label="Whales")
axes[0].set_title("Set duration after preprocessing")
axes[0].set_xlabel("seconds")
axes[0].set_ylabel("sets")
axes[0].legend()

axes[1].hist(lengths_50hz[mmfit_mask], bins=30, alpha=0.7, label="MM-Fit")
axes[1].hist(lengths_50hz[whales_mask], bins=30, alpha=0.7, label="Whales")
axes[1].set_title("50 Hz sequence lengths before 512 resample")
axes[1].set_xlabel("timesteps")
axes[1].legend()
plt.tight_layout()

## Normalization Checks

In [ ]:
train_mask = split == "train"
print("Scaler mean from metadata:", np.array(metadata["normalization"]["mean"]))
print("Scaler std from metadata:", np.array(metadata["normalization"]["std"]))
print("Train normalized channel means:", X[train_mask].reshape(-1, X.shape[-1]).mean(axis=0))
print("Train normalized channel stds:", X[train_mask].reshape(-1, X.shape[-1]).std(axis=0))

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.ravel()
for i, feature in enumerate(feature_names):
    ax = axes[i]
    ax.hist(X[train_mask, :, i].ravel(), bins=60, alpha=0.8)
    ax.set_title(str(feature))
    ax.set_yscale("log")
plt.suptitle("Train-split normalized channel distributions")
plt.tight_layout()

## Example Signals

In [ ]:
def plot_sample(idx: int):
    t = np.arange(X.shape[1]) / float(raw["target_hz"])
    fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
    axes[0].plot(t, X[idx, :, :3])
    axes[0].set_title(
        f"{idx}: {source[idx]} | {exercise_name[idx]} | reps={reps[idx]} | split={split[idx]} | acc"
    )
    axes[0].legend(feature_names[:3], ncol=3, loc="upper right")
    axes[1].plot(t, X[idx, :, 3:])
    axes[1].set_title("gyro")
    axes[1].legend(feature_names[3:], ncol=3, loc="upper right")
    axes[1].set_xlabel("resampled model time index / target_hz")
    plt.tight_layout()

# Plot one Whales supervised example from low, middle, and high rep counts.
for target_rep in [8, 12, 16]:
    candidates = np.flatnonzero(rep_supervised & (reps == target_rep))
    if candidates.size:
        plot_sample(int(candidates[0]))

## Training-Relevant Takeaways Checklist

Use this cell output to decide whether the dataset is ready for a training run.

- `rep_supervised` should be true only for Whales `8-16` examples.
- MM-Fit should stay useful for classification/segmentation supervision.
- Split-specific rep support should be inspected before interpreting exact accuracy.
- Baselines here must be copied into model reports before comparing learned rep-count metrics.

In [ ]:
checks = {
    "shape_is_780x512x6": X.shape == (780, 512, 6),
    "all_classification_supervised": bool(classification_supervised.all()),
    "rep_supervised_is_whales_only": bool(np.all(source[rep_supervised] == "whales")),
    "rep_classes_cover_0_to_8": set(rep_class[rep_supervised].tolist()) == set(range(9)),
    "no_nan_in_X": bool(np.isfinite(X).all()),
}
for name, ok in checks.items():
    print(f"{name}: {ok}")
assert all(checks.values()), checks